In [1]:
from pyspark.sql import SparkSession
import os

# Env vars you already use
ENDPOINT = os.getenv("DATA_LAKE_ENDPOINT")              # e.g. "http://minio:9000"
ACCESS_KEY = os.getenv("DATA_LAKE_ACCESS_KEY_ID")
SECRET_KEY = os.getenv("DATA_LAKE_SECRET_ACCESS_KEY")

# If your endpoint starts with http://, disable SSL in S3A
SSL_ENABLED = "true"
if ENDPOINT and ENDPOINT.startswith("http://"):
    SSL_ENABLED = "false"

spark = (
    SparkSession.builder
    .appName("Interactive Spark App")
    # ---- Delta Lake ----
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # ---- S3A / MinIO ----
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.endpoint", ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")  # REQUIRED for MinIO
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", SSL_ENABLED)
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    # Optional: helps with some S3-compatible quirks
    .config("spark.hadoop.fs.s3a.connection.maximum", "100")
    .getOrCreate()
)

spark


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/24 04:20:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
!env

MONGO_HOST=mongodb
LANGUAGE=en_US:en
MONGO_JAVA_DRIVER_VERSION=5.6.2
MPLBACKEND=module://matplotlib_inline.backend_inline
HOSTNAME=2b27df014a7c
SHLVL=0
HOME=/root
KAFKA_BOOTSTRAP=broker:29092
MONGO_PORT=27017
SPARK_BUFFER_SIZE=65536
GPG_KEY=F28C9C925C188C35E345614DEDA00CE834F0FC5C
PAGER=cat
SPARK_TGZ_ASC_URL=https://www.apache.org/dyn/closer.lua/spark/spark-4.0.1/spark-4.0.1-bin-hadoop3.tgz.asc?action=download
MONGO_SPARK_VERSION=10.5.0
KAFKA_VERSION=4.0.1
SPARK_TGZ_URL=https://www.apache.org/dyn/closer.lua/spark/spark-4.0.1/spark-4.0.1-bin-hadoop3.tgz?action=download
JAVA_VERSION=jdk-17.0.16+8
AWS_SDK_V1_VERSION=1.12.793
AWS_SDK_V2_VERSION=2.40.13
FORCE_COLOR=1
HADOOP_VERSION=3.4.0
MONGO_INITDB_ROOT_PASSWORD=password
SPARK_VERSION=4.0.1
TERM=xterm-color
MONGO_APP_DB=yelp
PATH=/opt/java/openjdk/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin
MONGO_AUTH_DB=admin
LANG=en_US.UTF-8
CLICOLOR_FORCE=1
DELTA_VERSION=4.0.0
SCALA_VERSION=2.13
GIT_PAGER=cat
JAVA_HOME=/opt/java/op

In [3]:
df_users = spark.read.format("delta").load('s3a://bronze/users')
df_users.printSchema()
print("Row count:", df_users.count())
df_users.show(20, truncate=False)

25/12/24 04:20:32 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


root
 |-- average_stars: double (nullable = true)
 |-- compliment_cool: long (nullable = true)
 |-- compliment_cute: long (nullable = true)
 |-- compliment_funny: long (nullable = true)
 |-- compliment_hot: long (nullable = true)
 |-- compliment_list: long (nullable = true)
 |-- compliment_more: long (nullable = true)
 |-- compliment_note: long (nullable = true)
 |-- compliment_photos: long (nullable = true)
 |-- compliment_plain: long (nullable = true)
 |-- compliment_profile: long (nullable = true)
 |-- compliment_writer: long (nullable = true)
 |-- cool: long (nullable = true)
 |-- elite: string (nullable = true)
 |-- fans: long (nullable = true)
 |-- friends: string (nullable = true)
 |-- funny: long (nullable = true)
 |-- name: string (nullable = true)
 |-- review_count: long (nullable = true)
 |-- useful: long (nullable = true)
 |-- user_id: string (nullable = true)
 |-- yelping_since: string (nullable = true)



25/12/24 04:20:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Row count: 994465


IOPub data rate exceeded.                                                       
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [4]:
df_users.rdd.getNumPartitions()

13

In [7]:
df_users.select('user_id', 'name', 'average_stars', 'yelping_since').show(10, truncate=False)

+----------------------+--------+-------------+-------------------+
|user_id               |name    |average_stars|yelping_since      |
+----------------------+--------+-------------+-------------------+
|j14WgRoU_-2ZE1aw1dXrJg|Daniel  |3.74         |2009-01-25 04:35:42|
|2WnXYQFK0hXEoTxPtV2zvg|Steph   |3.32         |2008-07-25 10:41:00|
|SZDeASXq7o05mMNLshsdIA|Gwen    |4.27         |2005-11-29 04:38:33|
|hA5lMy-EnncsH4JoR-hFGQ|Karen   |3.54         |2007-01-05 19:40:59|
|q_QQ5kBBwlCcbL1s4NVK3g|Jane    |3.85         |2005-03-14 20:26:35|
|cxuxXkcihfCbqt5Byrup8Q|Rob     |2.75         |2009-02-24 03:09:06|
|E9kcWJdJUHuTKfQurPljwA|Mike    |3.73         |2008-12-11 22:11:56|
|lO1iq-f75hnPNZkTy3Zerg|Rachelle|4.04         |2008-12-29 22:40:56|
|xoZvMJPDW6Q9pDAXI0e_Ww|Ryan    |3.89         |2009-05-27 06:12:10|
|fJZO_skqpnhk1kvomI4dmA|Jennifer|4.15         |2008-07-14 16:01:36|
+----------------------+--------+-------------+-------------------+
only showing top 10 rows


In [9]:
df_businesses = spark.read.format("delta").load('s3a://bronze/businesses')
df_businesses.printSchema()
print("Row count:", df_businesses.count())

root
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |    |-- BusinessParking: string (nullable = true)
 |    |-- ByAppointmentOnly: string (nullable = true)
 |    |-- Caters: string (nullable = true)
 |    |-- CoatCheck: string (nullable = true)
 |    |-- Corkage: string (nullable = true)
 |    |-- DietaryRestrictions: string (nullable = true)
 |    |-- DogsAllowed: string (nullable = true)
 |    |-- DriveThru: string (nullable = true)
 |    |-- GoodForDancing: str

[Stage 44:============================================>           (40 + 5) / 50]

Row count: 75391


In [14]:
df_businesses.select('business_id','name','address','attributes.RestaurantsAttire','stars','review_count').show(10, truncate=False)

+----------------------+--------------------------+---------------------------------+-----------------+-----+------------+
|business_id           |name                      |address                          |RestaurantsAttire|stars|review_count|
+----------------------+--------------------------+---------------------------------+-----------------+-----+------------+
|_7cf56aKblR62Jwor53Nlw|Circle K                  |1801 Concord Pike                |NULL             |3.5  |5           |
|T9xqrycUmpbwX_bMSo_vRw|Pretty Princess Jersey    |                                 |NULL             |4.0  |5           |
|1LoKM5J-w77CXL4jbdF2eA|Arch City Granite & Marble|1540 Page Industrial Blvd        |NULL             |3.0  |6           |
|qS9uPjY6Hkd8lEZk6cS2lw|Safelite AutoGlass        |3571 W Ina Rd, Ste 151           |NULL             |3.5  |49          |
|dBUlDcC7RZaqamswEm9Amw|Community Health Alliance |1055 S Wells Ave                 |NULL             |3.0  |27          |
|EpFFiPHKx9yP-Wq